# 01 -- Raster and Spatial Foundations

This notebook covers GeoTIFF I/O, terrain products, connected-region
analysis, and explicit grid alignment.  It draws from command-line examples
01 through 04.

**Run the individual scripts:** `01_geotiff_and_coordinates.py`,
`02_terrain_products.py`, `03_region_filtering.py`, `04_alignment.py`

## Setup

All examples use a deterministic 64x64 synthetic lunar DEM in a south-polar
stereographic projection.  Run this cell once before executing any of the
sections below.

In [ ]:
import sys, os
from pathlib import Path

def _repo_root():
    """Find the Lunarscout repository root from the kernel working directory."""
    for start in [Path.cwd()] + list(Path.cwd().parents):
        if (start / "src" / "lunarscout" / "__init__.py").exists():
            return start
    raise RuntimeError(
        "Cannot locate Lunarscout repository root. "
        "Launch Jupyter from the repository root directory."
    )

_REPO = _repo_root()
sys.path.insert(0, str(_REPO / "src"))
sys.path.insert(0, str(_REPO / "examples"))


import lunarscout as ls
import numpy as np

from _example_support import (
    ensure_synthetic_scenario,
    synthetic_georef,
    synthetic_dem,
)

WORKSPACE = Path("/tmp/lunarscout_notebook_01")
WORKSPACE.mkdir(parents=True, exist_ok=True)

ensure_synthetic_scenario(WORKSPACE)
SCENARIO = ls.open_scenario(WORKSPACE / "synthetic_scenario")

print(f"Workspace: {WORKSPACE}")
print(f"Scenario DEM: {SCENARIO.dem_path()}")

---
## 1. GeoTIFF Read, Inspect, and Write

Read the DEM, verify that it is georeferenced, inspect its metadata, and
write a copy.

In [ ]:
dem, georef = ls.read_geotiff(SCENARIO.dem_path())

if georef is None:
    raise RuntimeError("The DEM must be georeferenced.")

print(f"  dtype:           {dem.dtype}")
print(f"  shape:           {dem.shape}")
print(f"  min / max:       {dem.min():.2f} / {dem.max():.2f}")
print(f"  nodata:          {georef.nodata}")
print(f"  projection WKT:  {georef.projection_wkt[:60]}...")
print(f"  width x height:  {georef.width} x {georef.height}")
print(f"  pixel_size:      {georef.pixel_size_x:.2f} x {georef.pixel_size_y:.2f}")
print(f"  affine:          {georef.affine_transform}")

# Convert a sample pixel to projected and geographic coordinates.
x, y = 20, 16
px, py = georef.pixel_to_projected(x, y)
lon, lat = georef.pixel_to_lonlat(x, y)
print(f"\nPixel ({x},{y}) -> projected ({px:.2f}, {py:.2f}) m")
print(f"Pixel ({x},{y}) -> lon/lat   ({lon:.4f}, {lat:.4f}) deg")

In [ ]:
# Write a copy to the scenario analysis directory.
out = SCENARIO.output_path("analysis/dem_copy.tif")
ls.write_geotiff(out, dem, georef, overwrite=True)
print(f"Wrote {out}")

**Try this:** pick different pixel coordinates and convert them.  How do
projected coordinates change as you move one pixel right?  One pixel down?

---
## 2. Terrain Products -- Slope, Aspect, Hillshade

Lunarscout provides GDAL-compatible slope, aspect, and hillshade operations.
Each returns both the output values and a georeference for the product grid.

In [ ]:
slope_values, slope_georef = ls.slope(dem, georef, output_nodata=-9999.0)
aspect_values, aspect_georef = ls.aspect(dem, georef, output_nodata=-9999.0)
# Nodata pixels produce NaN intermediates that trigger a harmless
# NumPy cast warning; the function replaces them with output_nodata.
import warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="invalid value encountered in cast")
    hillshade_values, shade_georef = ls.hillshade(dem, georef, output_nodata=0)

valid_slope = slope_values[slope_values != slope_georef.nodata]
print(f"Slope     -- min: {valid_slope.min():.1f} deg, max: {valid_slope.max():.1f} deg")
valid_aspect = aspect_values[aspect_values != aspect_georef.nodata]
print(f"Aspect    -- min: {valid_aspect.min():.1f} deg, max: {valid_aspect.max():.1f} deg")
print(f"Hillshade -- min: {hillshade_values.min():.1f}, max: {hillshade_values.max():.1f}")

In [ ]:
_terrain = SCENARIO.output_path("analysis/terrain")
_terrain.mkdir(parents=True, exist_ok=True)
ls.write_geotiff(str(_terrain / "slope_deg.tif"), slope_values, slope_georef, overwrite=True)
ls.write_geotiff(str(_terrain / "aspect_deg.tif"), aspect_values, aspect_georef, overwrite=True)
ls.write_geotiff(str(_terrain / "hillshade.tif"), hillshade_values, shade_georef, overwrite=True)
print(f"Terrain products written under {_terrain}")

Hillshade is a visualisation aid; it does not represent physical
illumination.  Slope and aspect are quantitative terrain measurements.

**Try this:** change the hillshade azimuth and altitude keyword arguments
and re-run.

---
## 3. Connected Regions

Turn a slope-threshold into a candidate mask, then label, measure, filter,
and outline the connected regions.

In [ ]:
# Candidate: slope <= 8 degrees  (illustrative, not a mission threshold)
candidate = slope_values <= 8.0

labels, labels_georef = ls.label_regions(candidate, georef)
sizes, sizes_georef = ls.region_sizes(candidate, georef)

print(f"Number of connected candidates: {len(np.unique(labels)) - 1}")
print(f"Largest region size:            {sizes.max()}")

In [ ]:
# Keep regions with at least 80 cells, apply morphological opening.
large, large_georef = ls.filter_regions_by_size(
    candidate, georef,
    threshold=80, comparator=">=",
    cleanup="opening", iterations=1,
)

# Extract borders of the remaining regions.
borders, borders_georef = ls.find_borders(large, large_georef)

print(f"Regions kept after size filter: {large.sum()}")
print(f"Border cell count:              {borders.sum()}")

In [ ]:
_regions = SCENARIO.output_path("analysis/regions")
_regions.mkdir(parents=True, exist_ok=True)

# Use a zero-nodata encoding for boolean masks.
mask_nodata_georef = large_georef.with_nodata(0)
ls.write_geotiff(str(_regions / "large_regions.tif"), large.astype("uint8"), mask_nodata_georef, overwrite=True)
ls.write_geotiff(str(_regions / "borders.tif"), borders.astype("uint8"), mask_nodata_georef, overwrite=True)
print(f"Region products written under {_regions}")

The slope threshold of 8 degrees and the region-size threshold of 80 pixels
are **teaching choices**, not landing-site recommendations.

**Try this:** change the size threshold to 30 or 200.  Change the
comparator to `"<"`.  Use `connectivity=4` for cardinal-neighbor regions.

---
## 4. Grid Comparison and Explicit Alignment

Two arrays with the same shape can still describe different geographic
locations.  Lunarscout never infers grid compatibility from shape alone.

In [ ]:
# Create a shifted copy of the slope raster.
shifted_values = np.roll(slope_values, shift=1, axis=1)

# Build a georeference with the origin moved 5 metres east.
from dataclasses import replace
shifted_georef = replace(slope_georef,
    affine_transform=(
        slope_georef.affine_transform[0] + 5.0,
    ) + slope_georef.affine_transform[1:],
)

# Same shape, different grid.
print(f"Shapes match: {slope_values.shape == shifted_values.shape}")
print(f"Grids match:  {ls.same_grid(slope_georef, shifted_georef)}")

In [ ]:
# require_same_grid raises a structured error when grids differ.
try:
    ls.require_same_grid(slope_georef, shifted_georef)
except ls.GridMismatchError as e:
    print(f"GridMismatchError: code={e.code}")
    print(f"  Details: {e.details}")

In [ ]:
# Align the shifted raster onto the reference grid with bilinear resampling.
aligned, aligned_georef = ls.align(
    shifted_values, shifted_georef,
    to=slope_georef,
    resampling="bilinear",
    output_nodata=-9999.0,
)

ls.require_same_grid(aligned_georef, slope_georef)
print("After alignment: grids match")

_alignment = SCENARIO.output_path("analysis/alignment")
_alignment.mkdir(parents=True, exist_ok=True)
ls.write_geotiff(str(_alignment / "aligned.tif"), aligned, aligned_georef, overwrite=True)
print(f"Aligned raster written under {_alignment}")

Alignment is an explicit analysis step.  Choose nearest-neighbour for
categorical data and a continuous method (bilinear, cubic, lanczos) for
continuous measurements.

**Try this:** use `resampling="nearest"` and compare the aligned values.
How big are the differences?